In [ ]:
%pip --quiet install strauss
%pip install cdflib

!git clone https://github.com/rosedshepherd/soniTEA-STRAUSS.git



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
fatal: destination path 'soniTEA-STRAUSS' already exists and is not an empty directory.


In [ ]:
#strauss imports
from strauss.sonification import Sonification
from strauss.sources import Objects, Events
from strauss import channels
from strauss.score import Score
from strauss import sources as Sources
import numpy as np
from strauss.generator import Sampler, Synthesizer, Spectralizer
from strauss.tts_caption import render_caption

# other useful modules
import matplotlib.pyplot as plt
import IPython.display as ipd
#from IPython.core.display import display, Markdown, Latex, Image

import glob
import os
import copy
import pathlib
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup #Yeah, this is a very funny library name. No, not my idea
import cdflib
import urllib
import csv
from scipy import signal
from scipy.interpolate import interp1d


In [24]:
## Function to fetch the filenames associated with the supplied date
def find_VanAllen_files(File_Root,yyyymmdd,response):
    """
    Finds all files at the given root filename assciated with the supplied date
    at the supplied response location

    Parameters:
    -----------
    time : str
        Initial part of the given filenames one wants to find
    yyyymmdd : str
        The date of the desired file
    response : str
        Url location of where the files are to be found

    Returns:
    --------
    matching_files : array
        Array of filenames associated with the date
    """
    if response.status_code == 200:
        #Turn the web page into a searchable text format
        soup = BeautifulSoup(response.text, "html.parser")

        #"Please, O Glorious Python, find me files that start with this name"
        search_pattern = f"{File_Root}_{yyyymmdd}"

        # Use that name to find matching filenames in the list procured by Soup
        matching_files = [a.text for a in soup.find_all("a") if search_pattern in a.text]
        return matching_files
    else:
        print(f"Failed to access {File_Root} (Status code: {response.status_code})")
        return []

In [25]:
YEAR = '2013'
MONTH = '03' #Always use two digits, so for Jan it's 01, for Feb it's 02, etc.
DAY = '17'

# Combine the trings, Python is nice for this because it's like adding
# numbers...but words!
Chosen_Date = YEAR + MONTH + DAY

In [27]:
Num_Respository = 'https://cdaweb.gsfc.nasa.gov/pub/data/rbsp/rbspa/l4/emfisis/density/'+YEAR+'/'

LookHere = requests.get(Num_Respository)

NumFiles = find_VanAllen_files('rbsp-a_density_emfisis-l4',Chosen_Date,LookHere)

print('We found the following files for your date:\n',np.array(NumFiles).reshape(-1, 1))

FILENAME = NumFiles[0]

#Generate the URL of the data
Num_Directory = Num_Respository+FILENAME
#Retrieve data
urllib.request.urlretrieve(Num_Directory, 'data_file.cdf')
#Read cdf file
numdata = cdflib.CDF('data_file.cdf')
numdata.cdf_info()


bmag = numdata.varget('bmag')
ne = numdata.varget('density')
alpha = numdata.varget('wpe_over_wce')
nEpoch = numdata.varget('Epoch')
t_num = (nEpoch-nEpoch[0])/(nEpoch[-1]-nEpoch[0])*(24*3600)

We found the following files for your date:
 [['rbsp-a_density_emfisis-l4_20130317_v1.5.16.cdf']]


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)>